# 04 · Optuna hyperparameter optimisation
TPE sampler (seed 42), objective = 5-fold CV RMSE on the training split. The test split is never used during tuning.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
from src import config
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
M, F = config.METRICS_DIR, config.FIG_DIR
show = lambda name, w=900: display(Image(filename=str(F / name), width=w))

In [2]:
import joblib, matplotlib.pyplot as plt
from src.plotting import BLUE, ORANGE
rows = []
for t in ('total','per_ha'):
    for m in ('LightGBM','XGBoost'):
        b = json.loads((M/f'optuna_best_{t}_{m}.json').read_text())
        rows.append({'target': t, 'model': m, 'trials': b['n_trials'], 'best_cv_RMSE': round(b['best_cv_rmse'],2), 'seconds': b['seconds'], **b['best_params']})
pd.DataFrame(rows)

,target,model,trials,best_cv_RMSE,seconds,n_estimators,learning_rate,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda,min_child_weight
0,total,LightGBM,40,830.49,64.5,1100,0.008819,5.0,5,5.0,0.871094,0.998902,0.000002,0.001267,NaN
1,total,XGBoost,40,836.77,79.5,600,0.020207,NaN,2,NaN,0.773366,0.919197,3.997576,1.803115,28.223779
2,per_ha,LightGBM,40,200.23,61.8,850,0.005169,6.0,7,7.0,0.979331,0.919376,0.000136,0.020852,NaN
3,per_ha,XGBoost,40,201.44,52.0,450,0.010650,NaN,3,NaN,0.593787,0.927353,0.274649,0.025174,1.538623


In [3]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for i, t in enumerate(('total','per_ha')):
    for j, m in enumerate(('LightGBM','XGBoost')):
        s = joblib.load(config.MODELS_DIR/f'optuna_study_{t}_{m}.joblib')
        v = np.array([tr.value for tr in s.trials])
        ax = axes[i, j]
        ax.scatter(range(len(v)), v, s=14, color=BLUE, alpha=0.6, label='trial')
        ax.plot(np.minimum.accumulate(v), color=ORANGE, lw=2, label='best so far')
        ax.set(title=f'{m} — {t}', xlabel='trial', ylabel='CV RMSE', ylim=(v.min()*0.98, np.quantile(v, 0.9)*1.02)); ax.legend()
fig.tight_layout(); fig.savefig(F/'optuna_history.png', dpi=130)

<site-packages>\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from optuna.importance import get_param_importances
imp = {}
for t in ('total','per_ha'):
    for m in ('LightGBM','XGBoost'):
        s = joblib.load(config.MODELS_DIR/f'optuna_study_{t}_{m}.joblib')
        imp[f'{m}/{t}'] = get_param_importances(s)
pd.DataFrame(imp).round(3)

,LightGBM/total,XGBoost/total,LightGBM/per_ha,XGBoost/per_ha
learning_rate,0.363,0.247,0.227,0.080
colsample_bytree,0.330,0.130,0.017,0.189
reg_lambda,0.152,0.203,0.176,0.044
subsample,0.066,0.015,0.226,0.029
max_depth,0.032,0.098,0.030,0.261
min_child_samples,0.024,NaN,0.091,NaN
num_leaves,0.017,NaN,0.045,NaN
reg_alpha,0.012,0.231,0.169,0.085
n_estimators,0.004,0.052,0.020,0.132
min_child_weight,NaN,0.025,NaN,0.179


## Baseline vs tuned (CV on train)

In [5]:
for t in ('total','per_ha'):
    cv = pd.read_csv(M/f'cv_results_{t}.csv').set_index('model')
    display(Markdown(f'**{t}**'))
    display(cv.loc[['LightGBM','LightGBM (Optuna)','XGBoost','XGBoost (Optuna)'], ['cv_R2','cv_RMSE','cv_MAE']].round(4))

**total**

,cv_R2,cv_RMSE,cv_MAE
model,,,
LightGBM,0.9911,873.8117,638.6585
LightGBM (Optuna),0.9919,830.4915,600.7511
XGBoost,0.9906,896.4548,652.4566
XGBoost (Optuna),0.9918,836.7690,605.9629


**per_ha**

,cv_R2,cv_RMSE,cv_MAE
model,,,
LightGBM,0.4501,212.8858,168.5606
LightGBM (Optuna),0.5133,200.2297,158.4294
XGBoost,0.4182,219.0124,173.0529
XGBoost (Optuna),0.5074,201.4396,159.4027


## Findings

* 40 TPE trials per model per target (~1 min each on CPU). Optuna favoured **many shallow trees with a low learning rate** (LightGBM: 5–6 leaves, lr ≈ 0.005–0.009, 850–1,100 trees), consistent with a low-dimensional signal plus noise.
* Gains: per-ha LightGBM CV RMSE 212.9 → 200.2 (R² 0.450 → 0.513); XGBoost 219.0 → 201.4 (0.418 → 0.507). Total-yield LightGBM 873.8 → 830.5 kg (R² 0.9911 → 0.9919).
* Tuning closes the gap to the (untuned) sklearn GradientBoosting but improvements are modest — the problem is bounded by noise (ceiling ≈ 0.59), not by model capacity.
